<a href="https://colab.research.google.com/github/g25ai1020-lab/MLOps-Assignment/blob/main/project1_submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
SELECT
    venue_name,
    venue_capacity
FROM `bigquery-public-data.ncaa_basketball.mbb_teams`
WHERE market = 'Stanford'
LIMIT 1;


In [ ]:
SELECT
    COUNT(*) as games_count
FROM `bigquery-public-data.ncaa_basketball.mbb_games_sr`
WHERE venue_name = 'Maples Pavilion'
    AND season = 2013;

In [ ]:
SELECT *
FROM `bigquery-public-data.ncaa_basketball.mbb_teams`
LIMIT 1;

SELECT
  table_name,
  column_name
FROM `bigquery-public-data.ncaa_basketball.INFORMATION_SCHEMA.COLUMNS`
WHERE LOWER(column_name) LIKE '%color%'

ORDER BY table_name;

SELECT
    market,
    color
FROM `bigquery-public-data.ncaa_basketball.team_colors`
WHERE UPPER(SUBSTR(REPLACE(color, '#', ''), 1, 2)) = 'FF'
ORDER BY market;

In [ ]:
SELECT *
FROM `bigquery-public-data.ncaa_basketball.mbb_teams`
LIMIT 1;

SELECT
  table_name,
  column_name
FROM `bigquery-public-data.ncaa_basketball.INFORMATION_SCHEMA.COLUMNS`
WHERE LOWER(column_name) LIKE '%color%'

ORDER BY table_name;

SELECT
    market,
    color
FROM `bigquery-public-data.ncaa_basketball.team_colors`
WHERE UPPER(SUBSTR(REPLACE(color, '#', ''), 1, 2)) = 'FF'
ORDER BY market;

In [ ]:
SELECT
    COUNT(*) AS games_won,
    ROUND(AVG(h_points), 2) AS avg_stanford_score,
    ROUND(AVG(a_points), 2) AS avg_opponent_score
FROM `bigquery-public-data.ncaa_basketball.mbb_games_sr`
WHERE season BETWEEN 2013 AND 2017
  AND h_market = 'Stanford'
  AND h_points > a_points;

In [ ]:
SELECT
    COUNT(DISTINCT p.player_id) AS player_count
FROM `bigquery-public-data.ncaa_basketball.mbb_players_games_sr` p
JOIN `bigquery-public-data.ncaa_basketball.mbb_teams` t
    ON p.team_id = t.id
WHERE p.birthplace_city = t.venue_city
  AND p.birthplace_state = t.venue_state;

In [ ]:
SELECT
    win_name,
    lose_name,
    win_pts,
    lose_pts,
    (win_pts - lose_pts) AS margin
FROM `bigquery-public-data.ncaa_basketball.mbb_historical_tournament_games`
WHERE (win_pts - lose_pts) =
(
    SELECT MAX(win_pts - lose_pts)
    FROM `bigquery-public-data.ncaa_basketball.mbb_historical_tournament_games`
);

In [ ]:
SELECT
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*)
         FROM `bigquery-public-data.ncaa_basketball.mbb_historical_tournament_games`),
        2
    ) AS upset_percentage
FROM `bigquery-public-data.ncaa_basketball.mbb_historical_tournament_games`
WHERE win_seed > lose_seed;

In [ ]:
SELECT
    a.name AS team1,
    b.name AS team2,
    a.venue_state AS state
FROM `bigquery-public-data.ncaa_basketball.mbb_teams` a
JOIN `bigquery-public-data.ncaa_basketball.team_colors` ca
    ON a.id = ca.id
JOIN `bigquery-public-data.ncaa_basketball.mbb_teams` b
    ON a.venue_state = b.venue_state
JOIN `bigquery-public-data.ncaa_basketball.team_colors` cb
    ON b.id = cb.id
WHERE ca.color = cb.color
  AND a.name < b.name
ORDER BY team1;

In [ ]:
SELECT
    birthplace_city AS city,
    birthplace_state AS state,
    birthplace_country AS country,
    SUM(points) AS total_points
FROM `bigquery-public-data.ncaa_basketball.mbb_players_games_sr`
WHERE team_market = 'Stanford'
  AND season BETWEEN 2013 AND 2017
  AND birthplace_city IS NOT NULL
  AND birthplace_state IS NOT NULL
  AND birthplace_country IS NOT NULL
GROUP BY city, state, country
ORDER BY total_points DESC
LIMIT 3;

In [ ]:
WITH first_half_scores AS (
    SELECT
        game_id,
        player_id,
        team_id,
        SUM(points_scored) AS first_half_points
    FROM `bigquery-public-data.ncaa_basketball.mbb_pbp_sr`
    WHERE period = 1
    GROUP BY game_id, player_id, team_id
),
qualified_players AS (
    SELECT DISTINCT
        team_id,
        player_id
    FROM first_half_scores
    WHERE first_half_points >= 15
)
SELECT
    t.market,
    COUNT(DISTINCT q.player_id) AS player_count
FROM qualified_players q
JOIN `bigquery-public-data.ncaa_basketball.mbb_teams` t
    ON q.team_id = t.id
GROUP BY t.market
HAVING COUNT(DISTINCT q.player_id) > 5
ORDER BY player_count DESC, t.market
LIMIT 5;

In [ ]:
WITH season_wins AS (
    SELECT
        season,
        market,
        wins,
        MAX(wins) OVER(PARTITION BY season) AS max_wins
    FROM `bigquery-public-data.ncaa_basketball.mbb_historical_teams_seasons`
),
top_performers AS (
    SELECT
        season,
        market
    FROM season_wins
    WHERE wins = max_wins
)
SELECT
    market,
    COUNT(*) AS seasons_as_top_performer
FROM top_performers
WHERE market IS NOT NULL
GROUP BY market
ORDER BY seasons_as_top_performer DESC, market
LIMIT 5;

